# 01 — Baseline player detection

Run pretrained YOLO11 (COCO class 0 = person) on a clip and inspect:
- Mean detections per frame.
- Far-end recall (boxes whose midpoint y is in the top 40% of the frame).
- Bbox area distribution (catch noise like full-frame talking-head detections).

Drop a real video into `data/raw_clips/` first and update `VIDEO` below.

In [ ]:
import sys, statistics, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '..')
from config import Config
from src.detection import PlayerDetector

VIDEO = Path('../data/raw_clips/match_1080p.mp4')
START_FRAME = 4500   # skip intro footage
N_FRAMES = 60        # 1.2s @ 50fps — enough to spot the failure modes

In [ ]:
cfg = Config()
det = PlayerDetector(cfg)
cap = cv2.VideoCapture(str(VIDEO))
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
frames = []
for _ in range(N_FRAMES):
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
frame_h, frame_w = frames[0].shape[:2]
print(f'Loaded {len(frames)} frames, {frame_w}x{frame_h}')
all_dets = [det.detect(f, i) for i, f in enumerate(frames)]
counts = [len(d) for d in all_dets]
print(f'mean detections/frame: {statistics.mean(counts):.2f}  (min {min(counts)}, max {max(counts)})')

In [ ]:
areas = [((d.bbox[2]-d.bbox[0])*(d.bbox[3]-d.bbox[1]) / (frame_w*frame_h)) * 100
         for ds in all_dets for d in ds]
plt.hist(areas, bins=40)
plt.xlabel('bbox area (% of frame)'); plt.ylabel('count'); plt.title('Player bbox size distribution')
plt.show()

In [ ]:
far = sum(1 for ds in all_dets for d in ds if (d.bbox[1]+d.bbox[3])/2 < 0.4*frame_h)
near = sum(1 for ds in all_dets for d in ds if (d.bbox[1]+d.bbox[3])/2 >= 0.4*frame_h)
print(f'far-half detections: {far}\nnear-half detections: {near}')

In [ ]:
# Visual sanity check on a sample frame
sample = frames[len(frames)//2].copy()
for d in all_dets[len(all_dets)//2]:
    x1,y1,x2,y2 = (int(v) for v in d.bbox)
    cv2.rectangle(sample, (x1,y1), (x2,y2), (0,255,0), 2)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.show()